# Exploratory Data Analysis for Crop and Weather Data

## Gemini Notes

### Reference Chat

Refer to <https://gemini.google.com/app/858a45eb41b6ac2a> for details on focused crop and weather analysis.

Work backwards from this in the EDA.

### Take-away

#### Agricultural Calendar

<https://www.tanks-direct.co.uk/farming-calendar#:~:text=September,Weaning%20calves>

|Crop|Typical Drilling (Sowing)|Flowering (Anthesis)|Harvest
|-|-|-|-|
|Winter Wheat	|Sept – Oct		|June		|Aug – Sept
|Winter Barley	|Sept – early Oct|May – June	|July – Aug
|Spring Barley	|March – April	|June – July|Aug – Sept
|Oats (Winter)	|Sept – Oct		|June		|Aug
|Oilseed Rape	|August – early	Sept	|April – May	|July

#### Crop-Specific "Critical Windows"

##### Winter Wheat

Oct – Feb (Rainfall & Rain Days): Excess rain here causes waterlogging. Wheat has shallow roots early on; if the soil is anaerobic (no oxygen), yield potential is "capped" before spring even starts.

April – May (Air Frost): This is the "stem elongation" phase. A late air frost can damage the developing ear inside the stalk, leading to "blind" grain sites.

June (Sunshine & Max Temp): High sunshine in June is the strongest predictor of high yield (biomass accumulation). However, if Max Temp > 25-27°C, it can cause "heat stress," reducing the number of grains per ear.

##### Winter & Spring Barley

Sept – Oct (Rain Days): For Winter Barley, "Rain Days > 1mm" in autumn is a negative factor because it prevents timely drilling. Barley is less resilient than wheat to late sowing.

May – June (Rainfall & Min Temp): Barley needs "cool and moist" conditions during grain fill. Drought in May is the primary "yield killer" for Spring Barley, as it has a very short window to build biomass.

July (Rain Days): High rainfall during harvest leads to "lodging" (crops falling over) and sprouting, which destroys quality and yield.

#### "Gold Standard" Low-Noise

Suggestion:

|Factor				|Value			|
|-|-|
|Weather Region:	|East Anglia|
|Crop Region:		|Eastern England|
|Crop:				|Winter Wheat|
|Weather			|Type: Sunshine|
|Time Period:		|June|

Suggested stats, **Pearson Correlation** between **June** **Sunshine** and **Winter Wheat Yield**.

## [Contents](#contents)

- [1. Overview](#overview)
- [2. Environment Configuration](#2-environment-configuration)
- [3. Crops](#3-crops)
- [4. Weather](#4-weather)
- [5. Combined Weather 7 Crop Data Analysis](#5-combined-weather--crop-data-analysis)

## [2. Environment Configuration](#contents)

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
project_db  = 'data/project_db.db'

## [3. Crops](#contents)

The UK has a varied landscape with varying soil, weather, and topographic conditions. These could all be factors in crop yields. To reduce the influence of these, yield data for just East Anglia will be analysed.

### [3.1 Yearly Crop Yields](#contents)

#### Get Data

In [ ]:
qry_yields = """
	SELECT 
		o.year,
		o."United Kingdom" AS UK_Oats,
		osr."United Kingdom" AS UK_OSR,
		sb."United Kingdom" AS UK_Spring_Barley,
		w."United Kingdom" AS UK_Wheat,
		wb."United Kingdom" AS UK_Winter_Barley

	FROM
		yields_oats o

		JOIN yields_OSR osr
			ON o.year = osr.year

		JOIN yields_spring_barley sb
			ON o.year = sb.year

		JOIN yields_wheat w
			ON o.year = w.year

		JOIN yields_winter_barley wb
			ON o.year = wb.year

	WHERE
		o.year >= 1998 AND o.year < 2026
"""

conn = sqlite3.connect(project_db)

df_yields = pd.read_sql(qry_yields, conn)

conn.close()

df_yields.head(10)

#### Plot Yields over Time

In [ ]:
# define plot
plt.figure(figsize=(16,5))

# Create Month number field for x-axis and plot each year as a line on the same figure.
df.loc[:, 'Year'] = pd.to_datetime(df['Year'], format='%Y').dt.year
plt.plot(df_yields.Year, df_yields.UK_Oats , color='tab:red')
plt.plot(df_yields.Year, df_yields.UK_OSR , color='tab:blue')
plt.plot(df_yields.Year, df_yields.UK_Spring_Barley , color='tab:green')
plt.plot(df_yields.Year, df_yields.UK_Wheat , color='tab:purple')
plt.plot(df_yields.Year, df_yields.UK_Winter_Barley , color='tab:orange')


# add figure elements
plt.title(f'UK Annual Crop Yields (1999 - 2025)')
plt.xlabel('Year')
plt.ylabel('Rainfall (mm)')
plt.legend()
plt.show()

Observations:

- Yields evidently vary from year to year.
- These variations appear to be consistent across crops.
- Wheat is the highest yielding crop.

#### Yield Correlation Heatmap 

A correlation Heatmap will show if the yields are correlated.

In [ ]:
# Drop Year column
df_yields_corr = df_yields.drop(columns=['Year'], inplace=False)
df_yields_corr.head()

In [ ]:
# Assuming 'data' is your DataFrame
corr = df_yields_corr.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()

Observations:

- Wheat has high correlations with Winter Barley, Spring Barley and Oats.
- Oil Seed Rape (OSR) yield has little correlation with any other crop.

This suggests some underlying cause for at least Wheat, Spring Barley and Oats yields to fluctuate together. This could well be weather related.

#### Wheat Yield Variation Across Region

##### Get Regional Wheat Data

In [ ]:
qry_regional_wheat_yields = """
	SELECT 
		Year,
		England,
		Scotland,
		Wales,
		"Northern Ireland" AS NI

	FROM
		yields_wheat

	WHERE
		year >= 1998 AND year < 2026
"""

conn = sqlite3.connect(project_db)

df_wheat_yields = pd.read_sql(qry_regional_wheat_yields, conn)

conn.close()

df_wheat_yields.head(10)

##### Plot

In [ ]:
# define plot
plt.figure(figsize=(16,5))

df = df_wheat_yields

# Create Month number field for x-axis and plot each year as a line on the same figure.
df.loc[:, 'Year'] = pd.to_datetime(df['Year'], format='%Y').dt.year
plt.plot(df.Year, df.England , color='tab:red')
plt.plot(df.Year, df.Scotland , color='tab:orange')
plt.plot(df.Year, df.Wales , color='tab:green')
plt.plot(df.Year, df.NI , color='tab:blue')


# add figure elements
plt.title(f'UK Annual Crop Yields (1999 - 2025)')
plt.xlabel('Year')
plt.ylabel('Wheat Yield')
plt.legend(loc=0, ncols = 5)
plt.show()

## [4. Weather](#contents)

### [Rainfall](#contents)

#### 4.1 Rainfall Across Years

##### Get Data from database

In [ ]:
qry_uk_rf = """
	SELECT 
		year,
		jan,
		feb,
		mar,
		apr,
		may,
		jun,
		jul,
		aug,
		sep,
		oct,
		nov,
		dec

	FROM
		Rainfall
	WHERE
		year >= 1998 AND year < 2026
		AND 
		area = 'UK'
"""

conn = sqlite3.connect(project_db)

df_uk_rf = pd.read_sql(qry_uk_rf, conn)

conn.close()

df_uk_rf.head(10)

##### Pivot from Long to Short format dataframe for plotting

In [ ]:
# Un-pivot the data into long format.
df_uk_rf_long = df_uk_rf.melt(id_vars='year', var_name='month', value_name='rainfall')

# Create new field for date with month and year.
df_uk_rf_long['date'] = pd.to_datetime(df_uk_rf_long['year'].astype(str) + '-' + df_uk_rf_long['month'], format='%Y-%b')

# Sort data by the date and reset the index.
df_uk_rf_long = df_uk_rf_long.sort_values('date').reset_index(drop=True)

# Verify with sample.
df_uk_rf_long.head(14)

##### Timeseries Plot Rainfall between 1999 and 2025

In [ ]:
def plot_df(df, x, y, title="", xlabel='Date', ylabel='Value', dpi=100):
    plt.figure(figsize=(16,5), dpi=dpi)
    plt.plot(x, y, color='tab:red')
    plt.gca().set(title=title, xlabel=xlabel, ylabel=ylabel)
    plt.show()

plot_df(df_uk_rf_long, x=df_uk_rf_long.date, y=df_uk_rf_long.rainfall, title='UK Rainfall')

This figure shows some significant fluctuation. A close look at a few single years might show a pattern.

#### 4.2 Rainfall Overlay of Each Year by Month

##### Create Years of Interest List

In [ ]:
i = 1999
rf_years: list = []

while i < 2026:
	rf_years.append(i)
	i += 1

print(rf_years)

##### Dict of Dataframes for Each Year

In [ ]:
# List of years to add.
rf_dfs: dict = {}

# loop through all years in list and create dataframes.
for year in rf_years:
	df_name = f'df_rf_uk_{str(year)}'
	df_name = df_uk_rf_long[df_uk_rf_long['year'] == int(year)].copy()
	rf_dfs[str(year)] = df_name

	print(df_name.head(3))

##### Create Overlay Plot Month Rainfall for Each Year

In [ ]:
# define plot
plt.figure(figsize=(16,5))

# Create Month number field for x-axis and plot each year as a line on the same figure.
for year in rf_years:
	df = rf_dfs[str(year)]
	df.loc[:, 'month_num'] = pd.to_datetime(df['month'], format='%b').dt.month
	plt.plot(df['month_num'], df['rainfall'], label= str(year))


# add figure elements
plt.title(f'UK Monthly Rainfall ({rf_years[0]} - {rf_years[-1]})')
plt.xlabel('Month')
plt.ylabel('Rainfall (mm)')
plt.xticks(range(1,13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.legend(loc=0, ncols = 5)
plt.show()

This shows potential differences across the year for seasonal differences in rainfall. Depending on the sowing, growing and harvesting seasons of a crop, variations in more specific time periods could have a greater affect on yield than yearly averages.

#### 4.3 Rainfall Overlay of Each Year by Season

## [5. Combined Weather & Crop Data Analysis](#contents)

### 5.1 UK Correlation Heatmap

#### Pull Data

In [ ]:
query = """
	SELECT
		sb."Year" as year,
		sb."United Kingdom" AS uk_sb_yield,
		r.ann AS uk_rainfall,
		t.ann AS uk_max_temp,
		s.ann AS uk_sunshine

	FROM yields_spring_barley sb
		JOIN Rainfall r ON sb."Year" = r.year
		JOIN Tmax t ON sb."Year" = t.year
		JOIN Sunshine s ON sb."Year" = s.year
	
	WHERE r.area = 'UK'
		AND t.area = 'UK'
		AND s.area = 'UK'
		AND r.year >= 1999 AND r.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_annual_uk_sb = pd.read_sql(query, conn)

conn.close()

df_annual_uk_sb.head()

#### Remove Year

In [ ]:
df_annual_uk_sb_heatmap = df_annual_uk_sb.drop(columns=['year'], inplace=False)
df_annual_uk_sb_heatmap.head()

#### Plot Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame
corr = df_annual_uk_sb_heatmap.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()

### 5.2 East Anglia Correlation Heatmap

#### Pull data

In [ ]:
query = """
	SELECT
		sb."Year" as year,
		sb."Eastern" AS east_sb_yield,
		r.ann AS east_rainfall,
		t.ann AS east_max_temp,
		s.ann AS east_sunshine

	FROM yields_spring_barley sb
		JOIN Rainfall r ON sb."Year" = r.year
		JOIN Tmax t ON sb."Year" = t.year
		JOIN Sunshine s ON sb."Year" = s.year
	
	WHERE r.area = 'East_Anglia'
		AND t.area = 'East_Anglia'
		AND s.area = 'East_Anglia'
		AND r.year >= 1999 AND r.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_annual_east_sb = pd.read_sql(query, conn)

conn.close()

df_annual_east_sb.head()

#### Remove year column

In [ ]:
df_annual_east_sb_heatmap = df_annual_east_sb.drop(columns=['year'], inplace=False)
df_annual_east_sb_heatmap.head()

#### Plot Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame
corr = df_annual_east_sb_heatmap.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()